This script will first prepare a new SQLite database to hold the KSP, KYTC API and other reference data.  Next it will consolidate the annual KSP datasets into a single pandas dataframe and then create a new table within the database for later visualization use. Finally, it will load several reference datasets needed for visualization.


In [37]:
# Import modules
import os
from os.path import exists
import pandas as pd
import sqlite3
import time

Prior to setting up the database, we have manually downloaded the specific data from the Kentucky State Police public data access portal: http://crashinformationky.org/AdvancedSearch. Due to the lack of an API and constraints placed on downloads, the datasets were extracted as annual datasets in zipped csv files.  Each year's zipped dataset contain Incidents, TrafficControl, Person, AirBag, PropertyDamage, UnitFactors and Vehicle files.  For this data analysis, we will ingest incidents, traffic control, person, unit factors and vehicle data for 2020-2024. Note: 2024 is a partial year. As part of the initialization process, we define the path to the database and the raw crash data files to process and create the SQLite database.

In [38]:
# Define the path for the SQLite database
cwd = os.getcwd()
database_path = f'{cwd}/data/crash_data.db'
if exists(database_path):
    print("Database already exists")
else:
    os.makedirs(os.path.dirname(database_path), exist_ok=True)

# Define the path for the raw crash data files downloaded
directory_path = f'{cwd}/data/raw_crash_data'

# Create/Connect to SQLite database
conn = sqlite3.connect(database_path)
cursor = conn.cursor()

Database already exists


This following functions run a series of checks to prep the database if the whole process need to be rerun to ensure that the datasets are accurate.

In [39]:
def check_table_exists(database_path, table_name):
    query = f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';"
    with sqlite3.connect(database_path) as conn:
        cursor = conn.cursor()
        cursor.execute(query)
        result = cursor.fetchone()
    return result is not None

The following section was created to generate the list for the table creation step.  It is not necessary to run it each time, but it does not interfere with the overall process. To run the cell, uncomment the table you wish to check the schema for

In [40]:
# temp section to read the column names to create table columns
def read_column_names(csv_file_path):

    # Read only the first row of the CSV to get the column names
    df = pd.read_csv(csv_file_path, nrows=0)
    column_names = df.columns.tolist()
    return column_names

# Read the column names from the CSV fil- change as needed for different datasets
# csv_file_path = f'{cwd}/data/raw_crash_data/Incidents_2024.csv'
# csv_file_path = f'{cwd}/data/raw_crash_data/IncidentTrafficControl_2024.csv'
# csv_file_path = f'{cwd}/data/raw_crash_data/Person_2024.csv'
# csv_file_path = f'{cwd}/data/raw_crash_data/Vehicle_2024.csv'

# columns = read_column_names(csv_file_path)
# print(columns)

The following cell check if the table exists in the SQLite database - this is a temporary check and can be commented out when tested fully.  It will not interfere with the full process.

In [41]:
table_names = ['ksp_incidents', 'ksp_controls', 'ksp_person', 'ksp_vehicles', 'ksp_factors', 'county_district_lut', 'unit_factor_lut']

for table_name in table_names:
    if check_table_exists(database_path, table_name):
        print(f"The table '{table_name}' exists.")
        truncate_table(database_path, table_name)
        if check_table_has_data(database_path, table_name):
            print(f"The table '{table_name}' still has data after truncation.")
            delete_table(database_path, table_name)
            print(f"The table '{table_name}' has been deleted.")
        else:
            print(f"The table '{table_name}' is empty after truncation.")
    else:
        print(f"The table '{table_name}' does not exist.")

The table 'ksp_incidents' does not exist.
The table 'ksp_controls' does not exist.
The table 'ksp_person' does not exist.
The table 'ksp_vehicles' does not exist.
The table 'ksp_factors' does not exist.
The table 'county_district_lut' does not exist.
The table 'unit_factor_lut' does not exist.


Next we create the table to hold the incident schema in the database.  The KSP_Incidents table contains 4105 records.

In [42]:
# Create collision incidents table in database
cursor.execute('''CREATE TABLE IF NOT EXISTS ksp_incidents (
        IncidentID int,
        AgencyORI int,
        AgencyName TEXT,
        IncidentStatusDesc TEXT,
        County TEXT,
        RdwyNumber TEXT,
        Street TEXT,
        RoadwayName TEXT,
        StreetSfx TEXT,
        StreetDir TEXT,
        IntersectionRdwy TEXT,
        IntersectionRdwyName TEXT,
        BetweenStRdwy1 TEXT,
        BetweenStRdwyName1 TEXT,
        BetweenStRdwy2 TEXT,
        BetweenStRdwyName2 TEXT,
        Latitude REAL,
        Longitude REAL,
        Milepoint REAL,
        CollisionDate DATE,
        CollisionTime TIME,
        UnitsInvolved INT,
        MotorVehiclesInvolved INT,
        NumberKilled INT,
        NumberInjured INT,
        Weather TEXT,
        RdwyConditionCode INT,
        HitandRun TEXT,
        DirAnalysisCode	TEXT,
        MannerofCollision TEXT,
        RdwyCharacter TEXT,
        LightCondition TEXT,
        RampFromRdwyId TEXT,
        RampToRdwyId TEXT,
        AcceptedDate DATE,
        IsSecondaryCollision TEXT,
        OwnerBadge TEXT,
        IncidentStatus TEXT);''')

Next, we create a dataframe for each of the year's incidents and then concatenate them into a single dataframe.  To clean the data, we remove any "unnamed" columns that occur in the KSP downloaded csv files. One additional modification that is made is the standardization of the incident date values. If this step is not performed, further analysis of the data by date values is not possible.

In [43]:
# Append collision_incidents to dataframe
csv_files = [f for f in os.listdir(directory_path) if f.startswith("Incidents_") and f.endswith(".csv")]

# Initialize an empty list to hold dataframes
dataframes = []

# Iterate through the CSV files and load them into dataframes
for csv_file in csv_files:
    file_path = os.path.join(directory_path, csv_file)
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Concatenate all dataframes into a single dataframe
combined_incidents_df = pd.concat(dataframes, ignore_index=True)

# Drop any column with "Unnamed" in its name
unnamed_columns = [col for col in combined_incidents_df.columns if col.startswith('Unnamed')]
if unnamed_columns:
    combined_incidents_df = combined_incidents_df.drop(columns=unnamed_columns)

# Standardize the date column
combined_incidents_df['StandardizedCollisionDate'] = pd.to_datetime(
    combined_incidents_df['CollisionDate'], errors='coerce').dt.strftime('%Y-%m-%d')

# Drop the original CollisionDate column if needed
combined_incidents_df = combined_incidents_df.drop(columns=['CollisionDate'])

# Rename the new column to CollisionDate if necessary
combined_incidents_df = combined_incidents_df.rename(
    columns={'StandardizedCollisionDate': 'CollisionDate'})

# Set the option to display all columns
pd.set_option('display.max_columns', None)

print("All CSV files have been successfully loaded into a single DataFrame.")
print(combined_incidents_df)

All CSV files have been successfully loaded into a single DataFrame.
      IncidentID AgencyORI                     AgencyName IncidentStatusDesc  \
0       32655798    150000   BULLITT COUNTY SHERIFF DEPT.           Accepted   
1       32660760    568000   LOUISVILLE METRO POLICE DEPT           Accepted   
2       32660920    568000   LOUISVILLE METRO POLICE DEPT           Accepted   
3       32646189    150200  LEBANON JUNCTION POLICE DEPT.           Accepted   
4       32654634    930400     OLDHAM COUNTY POLICE DEPT.           Accepted   
...          ...       ...                            ...                ...   
4100    32686833   0930400     OLDHAM COUNTY POLICE DEPT.           Accepted   
4101    32672935   0150000   BULLITT COUNTY SHERIFF DEPT.           Accepted   
4102    32676655   0790000  MARSHALL COUNTY SHERIFF DEPT.           Accepted   
4103    32658708   0150000   BULLITT COUNTY SHERIFF DEPT.           Accepted   
4104    32659228   0150000   BULLITT COUNTY SHERIFF

/var/folders/64/k0s4v64x69z8dqjrj25vcvfh0000gn/T/ipykernel_50765/4292866678.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  combined_incidents_df['StandardizedCollisionDate'] = pd.to_datetime(


We want to save out the combined incident data in a single cleaned csv file in a subdirectory of the data folder

In [44]:
# Specify the path where you want to save the CSV file
output_path = f'{cwd}/data/clean_crash_data/collision_incidents.csv'

# Export the dataframe to a CSV file
combined_incidents_df.to_csv(output_path, index=False)
print(f"Dataframe exported successfully to {output_path}")

Dataframe exported successfully to /Users/terid/CodeYou_Capstone/data/clean_crash_data/collision_incidents.csv


Finally, the combined and clean incidents dataframe into a table within the SQLite database.

In [45]:
# Write the DataFrame to the SQLite table
combined_incidents_df.to_sql('ksp_incidents', conn, if_exists='append', index=False)

# Commit the changes and close the connection
conn.commit()
#conn.close() - only for the last dataset

print(f"Data from {df} has been successfully inserted into the collision_incidents table.")

Data from      IncidentID AgencyORI                     AgencyName IncidentStatusDesc  \
0      33431393   0568000   LOUISVILLE METRO POLICE DEPT           Accepted   
1      33430979   0250000     CLARK COUNTY SHERIFF DEPT.           Accepted   
2      33425916   0340200    LEXINGTON POLICE DEPARTMENT           Accepted   
3      33424176   0180000  CALLOWAY COUNTY SHERIFF DEPT.           Accepted   
4      33429942   0150100    SHEPHERDSVILLE POLICE DEPT.           Accepted   
..          ...       ...                            ...                ...   
435    32686833   0930400     OLDHAM COUNTY POLICE DEPT.           Accepted   
436    32672935   0150000   BULLITT COUNTY SHERIFF DEPT.           Accepted   
437    32676655   0790000  MARSHALL COUNTY SHERIFF DEPT.           Accepted   
438    32658708   0150000   BULLITT COUNTY SHERIFF DEPT.           Accepted   
439    32659228   0150000   BULLITT COUNTY SHERIFF DEPT.           Accepted   

        County RdwyNumber Street  Roadway

The next dataset will be the collision_traffic_controls.  This data indicates what type of traffic control was utilized by the work crews at the location of each incident. As with the previous dataset, the same process steps are followed. The KSP_Controls table contains 13492 records.

In [46]:
# Create table incident_traffic_control in database
cursor.execute('''CREATE TABLE IF NOT EXISTS ksp_controls (
        IncidentID int,
        TrafficControlNo int,
        TrafficControl TEXT);''')

In [47]:

# Combine all CSV files in the directory that begin with "IncidentTrafficControl_"
csv_files = [f for f in os.listdir(directory_path) if f.startswith("IncidentTraffic") and f.endswith(".csv")]

# Initialize an empty list to hold dataframes
dataframes = []

# Iterate through the CSV files and load them into dataframes
for csv_file in csv_files:
    file_path = os.path.join(directory_path, csv_file)
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Concatenate all dataframes into a single dataframe
combined_controls_df = pd.concat(dataframes, ignore_index=True)

#  Drop any column with "Unnamed" in its name
unnamed_columns = [col for col in combined_controls_df.columns if col.startswith('Unnamed')]
if unnamed_columns:
    combined_controls_df = combined_controls_df.drop(columns=unnamed_columns)

# Set the option to display all columns
pd.set_option('display.max_columns', None)

print("All CSV files have been successfully loaded into a single DataFrame.")
print(combined_controls_df)

All CSV files have been successfully loaded into a single DataFrame.
      IncidentId  TrafficControlNo       TrafficControl
0       26133281                 1     STOP & GO SIGNAL
1       26146419                 1  ADVISORY SPEED SIGN
2       26146419                 2          CENTER LINE
3       26146419                 3                OTHER
4       26146419                 4        WARNING SIGNS
...          ...               ...                  ...
6741    33431439                 3                OTHER
6742    33431439                 4        WARNING SIGNS
6743    33433119                 1               MEDIAN
6744    33433119                 2  ADVISORY SPEED SIGN
6745    33433119                 3        WARNING SIGNS

[6746 rows x 3 columns]


In [48]:

# Specify the path where you want to save the CSV file
output_path = f'{cwd}/data/clean_crash_data/incident_traffic_controls.csv'

# Export the dataframe to a CSV file
combined_controls_df.to_csv(output_path, index=False)

print(f"Dataframe exported successfully to {output_path}")

Dataframe exported successfully to /Users/terid/CodeYou_Capstone/data/clean_crash_data/incident_traffic_controls.csv


In [49]:
# Prepare the SQL insert statement dynamically based on DataFrame columns
columns = ', '.join([f'"{col}"' for col in combined_controls_df.columns])

placeholders = ', '.join(['?'] * len(combined_controls_df.columns))
sql = f'INSERT INTO ksp_controls ({columns}) VALUES ({placeholders})'

# Convert DataFrame to list of tuples
data_to_insert = combined_controls_df.to_records(index=False)

# Execute the SQL command using executemany
cursor.executemany(sql, data_to_insert)

# Commit changes and close the connection
conn.commit()

print("Data successfully added to the SQLite database at", database_path)

Data successfully added to the SQLite database at /Users/terid/CodeYou_Capstone/data/crash_data.db


In [50]:
# Write the DataFrame to the SQLite table
combined_controls_df.to_sql('ksp_controls', conn, if_exists='append', index=False)

# Commit the changes and close the connection
conn.commit()

print(f"Data from {df} has been successfully inserted into the collision_incidents table.")

Data from      IncidentId  TrafficControlNo       TrafficControl  Unnamed: 3
0      32658708                 1               MEDIAN         NaN
1      32659228                 1               MEDIAN         NaN
2      32672935                 1               MEDIAN         NaN
3      32676655                 1  ADVISORY SPEED SIGN         NaN
4      32676655                 2          CENTER LINE         NaN
..          ...               ...                  ...         ...
695    33431439                 3                OTHER         NaN
696    33431439                 4        WARNING SIGNS         NaN
697    33433119                 1               MEDIAN         NaN
698    33433119                 2  ADVISORY SPEED SIGN         NaN
699    33433119                 3        WARNING SIGNS         NaN

[700 rows x 4 columns] has been successfully inserted into the collision_incidents table.


Next we process the vehicles involved in each incident.  In addition to the make and model, the vehicles are classified into types, such as 'Passenger Car', 'Bus', etc. As with the previous datasets, the same process steps are followed. The KSP_Vehicles table contains 8205 records.

In [51]:
# Create table incident_vehicles in database
cursor.execute('''CREATE TABLE IF NOT EXISTS ksp_vehicles (
        IncidentID INT,
        UnitNumber INT,
        UnitType TEXT,
        AirbagSwitchCde TEXT,
        IsCommercialVeh TEXT,
        CrashAvoidCde TEXT,
        DriverIdentifiedCde TEXT,
        EventCollWithFirstCde TEXT,
        EventCollWithSecondCde TEXT,
        HasFire TEXT,
        PreCollActionCde TEXT,
        UnderOverrideCde TEXT,
        VehicleIsInsured TEXT,
        MakeCde TEXT,
        ModelCde TEXT,
        VehicleType TEXT,
        MakeDescription TEXT,
        ModelDescription TEXT);''')

In [52]:

# Combine all CSV files in the directory that begin with "Vehicle_"
csv_files = [f for f in os.listdir(directory_path)  if f.startswith("Vehicles_") and f.endswith(".csv")]

# Initialize an empty list to hold dataframes
dataframes = []

# Iterate through the CSV files and load them into dataframes
for csv_file in csv_files:
    file_path = os.path.join(directory_path, csv_file)
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Concatenate all dataframes into a single dataframe
combined_vehicles_df = pd.concat(dataframes, ignore_index=True)

# Drop any column with "Unnamed" in its name
unnamed_columns = [col for col in combined_vehicles_df.columns if col.startswith('Unnamed')]
if unnamed_columns:
    combined_vehicles_df = combined_vehicles_df.drop(columns=unnamed_columns)

# Set the option to display all columns
pd.set_option('display.max_columns', None)

print("All CSV files have been successfully loaded into a single DataFrame.")
print(combined_vehicles_df)

All CSV files have been successfully loaded into a single DataFrame.
      IncidentID  UnitNumber           UnitType  AirbagSwitchCde  \
0       33236305         1.0  HIT & RUN/UNKNOWN              NaN   
1       33207403         1.0  HIT & RUN/UNKNOWN              3.0   
2       33287817         1.0  HIT & RUN/UNKNOWN              3.0   
3       33307522         1.0  HIT & RUN/UNKNOWN              NaN   
4       33119026         1.0  HIT & RUN/UNKNOWN              1.0   
...          ...         ...                ...              ...   
8200    26916224         3.0      PASSENGER CAR              1.0   
8201    27128061         3.0  TRUCK-SINGLE UNIT              3.0   
8202    27276785         2.0  TRUCK-SINGLE UNIT              1.0   
8203    27364968         1.0    TRUCK & TRAILER              1.0   
8204    26960753         1.0  TRUCK-SINGLE UNIT              3.0   

     IsCommercialVeh  CrashAvoidCde DriverIdentifiedCde  \
0              False            NaN                  UN

In [53]:
# Specify the path where you want to save the CSV file
output_path = f'{cwd}/data/clean_crash_data/incident_vehicles.csv'

# Export the dataframe to a CSV file
combined_vehicles_df.to_csv(output_path, index=False)

print(f"Dataframe exported successfully to {output_path}")

Dataframe exported successfully to /Users/terid/CodeYou_Capstone/data/clean_crash_data/incident_vehicles.csv


In [54]:
# Write the DataFrame to the SQLite table
combined_vehicles_df.to_sql('ksp_vehicles', conn, if_exists='append', index=False)

# Commit the changes and close the connection
conn.commit()

print(f"Data from {df} has been successfully inserted into the collision_incidents table.")

Data from       IncidentID  UnitNumber                          UnitType  \
0       26781816         1.0  FARM TRACTOR &/OR FARM EQUIPMENT   
1       26757618         1.0                            GOCART   
2       26791538         2.0                 HIT & RUN/UNKNOWN   
3       26921560         1.0                 HIT & RUN/UNKNOWN   
4       26148508         1.0                 HIT & RUN/UNKNOWN   
...          ...         ...                               ...   
1460    26916224         3.0                     PASSENGER CAR   
1461    27128061         3.0                 TRUCK-SINGLE UNIT   
1462    27276785         2.0                 TRUCK-SINGLE UNIT   
1463    27364968         1.0                   TRUCK & TRAILER   
1464    26960753         1.0                 TRUCK-SINGLE UNIT   

      AirbagSwitchCde IsCommercialVeh  CrashAvoidCde DriverIdentifiedCde  \
0                 NaN           False            NaN                  UN   
1                 3.0           False        

The next dataset to be processed is related to the individual people involved in each incident.  As with the previous dataset, the same process steps are followed. The KSP_Person table contains 15328 records.

In [55]:
# Create table ksp_person in database
cursor.execute('''CREATE TABLE IF NOT EXISTS ksp_person (
        IncidentID INT,
        UnitNumber INT,
        PersonNo INT,
        PersonTypeCde TEXT,
        DeathDte DATE,
        AgeAtIncident INT,
        Gender TEXT,
        IsOwner TEXT,
        WasTransported TEXT,
        InjurySeverityCde TEXT,
        InjuryLocationCde TEXT,
        PosInVehicleCde TEXT,
        RestraintUseCde TEXT,
        TrappedCde TEXT,
        EjectionCde TEXT,
        EjectionPathCde TEXT,
        SuspectedOfDrinking TEXT,
        TestOffered TEXT,
        TestRefused TEXT,
        TestedForCde TEXT,
        TestSentTo TEXT,
        TestResults TEXT,
        HasOpLicense TEXT,
        HasCDLicense TEXT,
        HasLicenseRestrictions TEXT,
        HasOpEndorsements TEXT);''')

In [56]:
# Combine all CSV files in the directory that begin with "IncidentTrafficControl_"
csv_files = [f for f in os.listdir(directory_path)  if f.startswith("Person_") and f.endswith(".csv")]
print(csv_files)

# Initialize an empty list to hold dataframes
dataframes = []

# Iterate through the CSV files and load them into dataframes
for csv_file in csv_files:
    file_path = os.path.join(directory_path, csv_file)
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Concatenate all dataframes into a single dataframe
combined_person_df = pd.concat(dataframes, ignore_index=True)

# Drop any column with "Unnamed" in its name
unnamed_columns = [col for col in combined_person_df.columns if col.startswith('Unnamed')]
if unnamed_columns:
    combined_person_df = combined_person_df.drop(columns=unnamed_columns)

# Set the option to display all columns
pd.set_option('display.max_columns', None)

print("All CSV files have been successfully loaded into a single DataFrame.")
print(combined_person_df)

['Person_2024.csv', 'Person_2022.csv', 'Person_2023.csv', 'Person_2021.csv', 'Person_2020.csv']
All CSV files have been successfully loaded into a single DataFrame.
       IncidentID  UnitNumber  PersonNo PersonTypeCde DeathDte AgeAtIncident  \
0      32682385.0         2.0         3             8      NaN           NaN   
1      32686843.0         2.0         2             8      NaN           NaN   
2      32700076.0         NaN         2             9      NaN           NaN   
3      32702194.0         2.0         2             8      NaN           NaN   
4      32709983.0         1.0         2             8      NaN           NaN   
...           ...         ...       ...           ...      ...           ...   
15323         2.0         2.0         1           NaN     37.0          MALE   
15324         3.0         3.0         1           NaN     67.0          MALE   
15325         1.0         2.0         8           NaN      NaN       UNKNOWN   
15326         1.0         2.0      

In [57]:
# Specify the path where you want to save the CSV file for ksp person data
output_path = f'{cwd}/data/clean_crash_data/ksp_person.csv'

# Export the dataframe to a CSV file
combined_person_df.to_csv(output_path, index=False)

print(f"Dataframe exported successfully to {output_path}")

Dataframe exported successfully to /Users/terid/CodeYou_Capstone/data/clean_crash_data/ksp_person.csv


In [58]:
# Write the KSP_Person DataFrame to the SQLite table
combined_person_df.to_sql('ksp_person', conn, if_exists='append', index=False)

# Commit the changes and close the connection
conn.commit()

print(f"Data from {df} has been successfully inserted into the collision_incidents table.")

Data from           IncidentID  UnitNumber  PersonNo PersonTypeCde  DeathDte  \
26133281         1.0           3         8           NaN       NaN   
26146419         1.0           2         8           NaN       NaN   
26146419         2.0           4         8           NaN       NaN   
26148508         2.0           2         8           NaN       NaN   
26166091         2.0           4         8           NaN       NaN   
...              ...         ...       ...           ...       ...   
32884044         2.0           2         1           NaN      37.0   
32884044         3.0           3         1           NaN      67.0   
27255640         1.0           2         8           NaN       NaN   
26825719         1.0           2         8           NaN       NaN   
26403706         1.0           2         8           NaN       NaN   

         AgeAtIncident Gender IsOwner  WasTransported  InjurySeverityCde  \
26133281           NaN   True   False             NaN                NaN 

The Unit Factor datasets describe what factors might have contributed to the incident.  As with the previous datasets, the same process steps are followed. The Unit_Factor table contains 27282 records.

In [59]:
# Create table incident_vehicles in database
cursor.execute('''CREATE TABLE IF NOT EXISTS ksp_factors (
        IncidentID INT,
        UnitNumber INT,
        Factor_Type TEXT,
        Factor TEXT);''')

In [60]:
# Combine all CSV files in the directory that begin with "Unit_Factors_"
csv_files = [f for f in os.listdir(directory_path)  if f.startswith("Unit_") and f.endswith(".csv")]

# Initialize an empty list to hold dataframes
dataframes = []

# Iterate through the CSV files and load them into dataframes
for csv_file in csv_files:
    file_path = os.path.join(directory_path, csv_file)
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Concatenate all dataframes into a single dataframe
combined_factors_df = pd.concat(dataframes, ignore_index=True)

# Drop any column with "Unnamed" in its name
unnamed_columns = [col for col in combined_factors_df.columns if col.startswith('Unnamed')]
if unnamed_columns:
    combined_factors_df = combined_factors_df.drop(columns=unnamed_columns)

# Set the option to display all columns
pd.set_option('display.max_columns', None)

print("All CSV files have been successfully loaded into a single DataFrame.")
print(combined_factors_df)

All CSV files have been successfully loaded into a single DataFrame.
       IncidentId  UnitNumber           Factor_Type  Factor
0        32658708           1        ENVIRON FACTOR       2
1        32658708           2        ENVIRON FACTOR       2
2        32658708           3        ENVIRON FACTOR       2
3        32659228           1        ENVIRON FACTOR       2
4        32659228           2        ENVIRON FACTOR       2
...           ...         ...                   ...     ...
27277    30089216           1  DRIVER DISTRACTED BY       3
27278    30329814           1  DRIVER DISTRACTED BY       3
27279    29978338           1  DRIVER DISTRACTED BY       4
27280    30141594           1  DRIVER DISTRACTED BY       4
27281    29915633           1  DRIVER DISTRACTED BY       1

[27282 rows x 4 columns]


In [61]:
# Specify the path where you want to save the CSV file
output_path = f'{cwd}/data/clean_crash_data/incident_factors.csv'

# Export the dataframe to a CSV file
combined_factors_df.to_csv(output_path, index=False)

print(f"Dataframe exported successfully to {output_path}")

Dataframe exported successfully to /Users/terid/CodeYou_Capstone/data/clean_crash_data/incident_factors.csv


In [62]:

# Write the DataFrame to the SQLite table
combined_factors_df.to_sql('ksp_factors', conn, if_exists='append', index=False)

# Commit the changes and close the connection
conn.commit()

print(f"Data from {df} has been successfully inserted into the ksp_factors table.")

Data from       IncidentId  UnitNumber           Factor_Type  Factor  Unnamed: 4
0       29540000           1        ENVIRON FACTOR       2         NaN
1       29540000           1        ENVIRON FACTOR       6         NaN
2       29540000           2        ENVIRON FACTOR      99         NaN
3       29556449           1        ENVIRON FACTOR       2         NaN
4       29556449           1        ENVIRON FACTOR      11         NaN
...          ...         ...                   ...     ...         ...
5416    30089216           1  DRIVER DISTRACTED BY       3         NaN
5417    30329814           1  DRIVER DISTRACTED BY       3         NaN
5418    29978338           1  DRIVER DISTRACTED BY       4         NaN
5419    30141594           1  DRIVER DISTRACTED BY       4         NaN
5420    29915633           1  DRIVER DISTRACTED BY       1         NaN

[5421 rows x 5 columns] has been successfully inserted into the ksp_factors table.


There are additional lookup tables needed to allow for some of the data analysis.  The first is a County-District Lookup table.  This table, when joined to the incident's county name, a KYTC district is applied.  KYTC has 12 districts responsible for maintenance and operation of the highways of the Commonwealth. County_District_Lookup table contains 120 records.

In [63]:
# Change directory path to reference data
# Define the path for the raw crash data files downloaded
directory_path = f'{cwd}/data/reference_data'

In [64]:
# Create table incident_vehicles in database
cursor.execute('''CREATE TABLE IF NOT EXISTS county_district_lut (
        OBJECTID INT,
        Cnty_Name_UC TEXT,
        Cnty_Name_PC TEXT,
        Cnty_Number INT,
        Cnty_FIPS_Number INT,
        KYTC_District_Number INT,
        D_DISTRICT TEXT
        );''')

In [65]:
csv_file = directory_path + '/county_lut.csv'
file_path = os.path.join(directory_path, csv_file)
df = pd.read_csv(file_path)

In [66]:
# Write the DataFrame to the SQLite table
df.to_sql('county_district_lut', conn, if_exists='append', index=False)

# Commit the changes and close the connection
conn.commit()

print(f"Data from {df} has been successfully inserted into the county_district_lut table.")

Data from      OBJECTID Cnty_Name_UC Cnty_Name_PC  Cnty_Number  Cnty_FIPS_Number  \
0           1        ADAIR        Adair            1                 1   
1           2        ALLEN        Allen            2                 3   
2           3     ANDERSON     Anderson            3                 5   
3           4      BALLARD      Ballard            4                 7   
4           5       BARREN       Barren            5                 9   
..        ...          ...          ...          ...               ...   
115       116        WAYNE        Wayne          116               231   
116       117      WEBSTER      Webster          117               233   
117       118      WHITLEY      Whitley          118               235   
118       119        WOLFE        Wolfe          119               237   
119       120     WOODFORD     Woodford          120               239   

     KYTC_District_Number     D_DISTRICT  
0                       8       Somerset  
1              

The second lookup table we upload is for the descriptions of the unit factor codes that are numeric code numbers.  To create the Word Cloud properly, we want the user-friendly descriptions to join to the unit factor table. Factor_Code_LUT has 26 records.

In [67]:
# Create table incident_Factors in database
cursor.execute('''CREATE TABLE IF NOT EXISTS unit_factor_code_lut (
        Factor_code TEXT,
        Description TEXT
        );''')

In [68]:
csv_file = directory_path + '/factor_code_lut.csv'
file_path = os.path.join(directory_path, csv_file)
df = pd.read_csv(file_path)

In [69]:
# Write the DataFrame to the SQLite table
df.to_sql('unit_factor_code_lut', conn, if_exists='append', index=False)

# Commit the changes and close the connection
conn.commit()
conn.close()

print(f"Data from {df} has been successfully inserted into the unit factor code lookup table.")

Data from     Factor_Code                   Description
0             1           Alcohol Involvement
1             2                    Cell Phone
2             3     Disregard Traffic Control
3             4                   Distraction
4             5              Drug Involvement
5             6                     Emotional
6             7   Exceeded Stated Speed Limit
7             8  Failed to Yield Right of Way
8             9                       Fatigue
9            10                   Fell Asleep
10           11           Following Too Close
11           12              Improper Backing
12           13              Improper Passing
13           14                   Inattention
14           15    Lost Consciousness/Fainted
15           16                    Medication
16           17            Misjudge Clearnace
17           18      Not Under Proper Control
18           19   Overcorrecting/Oversteering
19           20           Physical Disability
20           21         